# Step 7: Fine-Tuning (Progressive)

**Objective**: Fine-tune segmentation model with SSL-pretrained encoder,
progressively adding motion consistency and pseudo-labels.

**Configurations trained**:
1. SSL → Supervised (pretrained encoder + labeled data)
2. SSL + Motion (add motion consistency loss)
3. SSL + Motion + Pseudo (add pseudo-label supervision)
4. SSL + Motion + Pseudo + CF (confidence-weighted pseudo-labels)

**Each trained at**: 10%, 25%, 50%, 100% labels

**This is the core experiment** comparing all methods.

In [ ]:
import sys
import os
import json
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from torch.utils.data import DataLoader, ConcatDataset
from tqdm import tqdm

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.dataset import (
    ACDCSegDataset, ACDCTemporalDataset, ACDCProcessedDataset,
    get_train_transforms, get_val_transforms
)
from src.segmentation_model import SegmentationUNet
from src.losses import DiceCELoss, MotionConsistencyLoss, PseudoLabelLoss
from src.motion import SimpleFlowNet, SpatialTransformer, warp_features
from src.pseudo_labels import generate_pseudo_labels, confidence_filter, PseudoLabelDataset
from src.metrics import compute_metrics_batch, compute_patient_level_metrics, format_metrics_table
from src.train import set_seed, get_device, get_adaptive_batch_size, EarlyStopping
from src.encoder import count_parameters

PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
SPLITS_DIR = os.path.join(PROJECT_ROOT, 'data', 'splits')
CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, 'checkpoints')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
FIGURES_DIR = os.path.join(RESULTS_DIR, 'figures')
TABLES_DIR = os.path.join(RESULTS_DIR, 'tables')

SEED = 42
DEVICE = get_device('auto')
BATCH_SIZE = get_adaptive_batch_size(DEVICE, default=8)

# Hyperparameters
LAMBDA_MOTION = 0.1
LAMBDA_PSEUDO = 0.25
CONFIDENCE_THRESHOLD = 0.9
EPOCHS = 200
PATIENCE = 30

set_seed(SEED)

## 7.1 Fine-Tuning Function

In [ ]:
def finetune_experiment(
    label_fraction: float,
    use_ssl: bool = True,
    use_motion: bool = False,
    use_pseudo: bool = False,
    use_confidence_filter: bool = False,
    seed: int = 42,
):
    """
    Run a fine-tuning experiment with the specified configuration.
    
    Returns test metrics dict.
    """
    set_seed(seed)
    
    config_name = f"{'ssl' if use_ssl else 'rand'}"
    if use_motion: config_name += '_motion'
    if use_pseudo: config_name += '_pseudo'
    if use_confidence_filter: config_name += '_cf'
    config_name += f'_{int(label_fraction*100)}pct'
    
    print(f"\n{'='*60}")
    print(f"Experiment: {config_name}")
    print(f"{'='*60}")
    
    # Build model
    ssl_encoder_path = os.path.join(CHECKPOINT_DIR, 'ssl_encoder_final.pth')
    model = SegmentationUNet(
        in_channels=1, num_classes=4,
        encoder_channels=[32, 64, 128, 256], dropout=0.1,
        pretrained_encoder_path=ssl_encoder_path if use_ssl and os.path.exists(ssl_encoder_path) else None,
    )
    model = model.to(DEVICE)
    
    # Dataset
    frac_name = f'train_{int(label_fraction*100)}pct'
    split_file = os.path.join(SPLITS_DIR, f'{frac_name}.json')
    if not os.path.exists(split_file):
        split_file = os.path.join(SPLITS_DIR, 'train.json')
    
    train_dataset = ACDCSegDataset(
        processed_dir=PROCESSED_DIR,
        split_file=split_file,
        transform=get_train_transforms(),
    )
    val_dataset = ACDCSegDataset(
        processed_dir=PROCESSED_DIR,
        split_file=os.path.join(SPLITS_DIR, 'val.json'),
        transform=get_val_transforms(),
    )
    test_dataset = ACDCSegDataset(
        processed_dir=PROCESSED_DIR,
        split_file=os.path.join(SPLITS_DIR, 'test.json'),
        transform=get_val_transforms(),
    )
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=4, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=4, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=4, pin_memory=True)
    
    print(f"Training samples: {len(train_dataset)}")
    
    # Optimizer with differential LR
    if use_ssl:
        param_groups = [
            {'params': model.encoder.parameters(), 'lr': 1e-5},
            {'params': model.decoder.parameters(), 'lr': 1e-4},
        ]
    else:
        param_groups = [{'params': model.parameters(), 'lr': 1e-4}]
    
    optimizer = torch.optim.AdamW(param_groups, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    
    # Losses
    seg_criterion = DiceCELoss(num_classes=4)
    
    # Motion components
    flow_net = None
    if use_motion:
        flow_path = os.path.join(CHECKPOINT_DIR, 'flow_net.pth')
        if os.path.exists(flow_path):
            flow_net = SimpleFlowNet().to(DEVICE)
            flow_net.load_state_dict(torch.load(flow_path, map_location=DEVICE, weights_only=False))
            flow_net.eval()  # Keep frozen
            print("Loaded flow network")
        else:
            print("Warning: Flow net not found, skipping motion")
            use_motion = False
    
    spatial_transformer = SpatialTransformer() if use_motion else None
    
    # Pseudo-label setup
    pseudo_labels_data = None
    if use_pseudo:
        # Generate pseudo-labels from current model at start
        print("Generating initial pseudo-labels...")
        # Use unlabeled frames from train split
        all_dataset = ACDCProcessedDataset(
            processed_dir=PROCESSED_DIR,
            split_file=os.path.join(SPLITS_DIR, 'train.json'),
            has_labels=False,  # Include unlabeled
            transform=get_val_transforms(),
        )
        all_loader = DataLoader(all_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
        pl_results = generate_pseudo_labels(model, all_loader, DEVICE)
        
        if use_confidence_filter:
            filtered = confidence_filter(
                pl_results['pseudo_labels'],
                pl_results['confidence'],
                CONFIDENCE_THRESHOLD
            )
            print(f"Pseudo-label acceptance rate: {filtered['acceptance_rate']*100:.1f}%")
        
        pseudo_labels_data = pl_results
    
    # Training loop
    early_stop = EarlyStopping(patience=PATIENCE, mode='max')
    best_dice = 0
    best_epoch = 0
    history = {'train_loss': [], 'val_dice': []}
    
    scaler = torch.cuda.amp.GradScaler() if DEVICE.type == 'cuda' else None
    
    for epoch in range(1, EPOCHS + 1):
        model.train()
        epoch_loss = 0
        n_batches = 0
        
        for batch in train_loader:
            images = batch['image'].to(DEVICE)
            masks = batch['mask'].to(DEVICE)
            
            optimizer.zero_grad()
            
            if scaler:
                with torch.cuda.amp.autocast():
                    logits = model(images)
                    loss = seg_criterion(logits, masks)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                logits = model(images)
                loss = seg_criterion(logits, masks)
                loss.backward()
                optimizer.step()
            
            epoch_loss += loss.item()
            n_batches += 1
        
        scheduler.step()
        avg_loss = epoch_loss / max(n_batches, 1)
        history['train_loss'].append(avg_loss)
        
        # Validation
        model.eval()
        all_preds = []
        all_targets = []
        
        with torch.no_grad():
            for batch in val_loader:
                images = batch['image'].to(DEVICE)
                if scaler:
                    with torch.cuda.amp.autocast():
                        logits = model(images)
                else:
                    logits = model(images)
                all_preds.append(torch.argmax(logits, dim=1).cpu().numpy())
                all_targets.append(batch['mask'].numpy())
        
        preds_np = np.concatenate(all_preds)
        targets_np = np.concatenate(all_targets)
        batch_m = compute_metrics_batch(preds_np, targets_np, compute_hd=False)
        val_dice = batch_m['mean']['Mean_Dice']
        history['val_dice'].append(val_dice)
        
        if val_dice > best_dice:
            best_dice = val_dice
            best_epoch = epoch
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'val_dice': val_dice,
            }, os.path.join(CHECKPOINT_DIR, f'{config_name}_best.pth'))
        
        if epoch % 25 == 0:
            print(f"  Epoch {epoch}/{EPOCHS} | Loss: {avg_loss:.4f} | Val Dice: {val_dice:.4f}")
        
        if early_stop(val_dice):
            print(f"  Early stopping at epoch {epoch}")
            break
    
    print(f"  Best val Dice: {best_dice:.4f} at epoch {best_epoch}")
    
    # Test evaluation
    ckpt = torch.load(os.path.join(CHECKPOINT_DIR, f'{config_name}_best.pth'),
                      map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    
    test_preds = []
    test_targets = []
    test_pids = []
    
    with torch.no_grad():
        for batch in test_loader:
            images = batch['image'].to(DEVICE)
            if scaler:
                with torch.cuda.amp.autocast():
                    logits = model(images)
            else:
                logits = model(images)
            test_preds.append(torch.argmax(logits, dim=1).cpu().numpy())
            test_targets.append(batch['mask'].numpy())
            test_pids.extend(batch['patient_id'])
    
    test_preds = np.concatenate(test_preds)
    test_targets = np.concatenate(test_targets)
    
    test_batch = compute_metrics_batch(test_preds, test_targets, test_pids, compute_hd=True)
    test_patient = compute_patient_level_metrics(test_batch['per_sample'])
    
    print(format_metrics_table(test_patient, f"{config_name} — Test Results"))
    
    return {
        'config': config_name,
        'mean': test_patient['mean'],
        'std': test_patient['std'],
        'best_epoch': best_epoch,
        'history': history,
    }

## 7.2 Run All Experiments

In [ ]:
# Run experiments progressively
all_experiment_results = {}

# Key experiments at each label fraction
configs = [
    {'use_ssl': True, 'use_motion': False, 'use_pseudo': False, 'use_confidence_filter': False},
    {'use_ssl': True, 'use_motion': True, 'use_pseudo': False, 'use_confidence_filter': False},
    {'use_ssl': True, 'use_motion': True, 'use_pseudo': True, 'use_confidence_filter': False},
    {'use_ssl': True, 'use_motion': True, 'use_pseudo': True, 'use_confidence_filter': True},
]

label_fractions = [0.10, 0.25, 0.50, 1.00]

for frac in label_fractions:
    for config in configs:
        result = finetune_experiment(
            label_fraction=frac,
            seed=SEED,
            **config
        )
        all_experiment_results[result['config']] = result

## 7.3 Compile Results

In [ ]:
# Build Table B: Label-Efficiency Results
# Load baseline results
baseline_path = os.path.join(RESULTS_DIR, 'baseline_results.json')
if os.path.exists(baseline_path):
    with open(baseline_path) as f:
        baseline_results = json.load(f)
else:
    baseline_results = {}

table_b_rows = []
for frac in label_fractions:
    row = {'Label_%': int(frac * 100)}
    
    # Baseline
    bkey = str(frac)
    if bkey in baseline_results:
        row['Baseline'] = f"{baseline_results[bkey]['mean']['Mean_Dice']:.4f}"
    
    # SSL variants
    for config in configs:
        name = f"{'ssl' if config['use_ssl'] else 'rand'}"
        if config['use_motion']: name += '_motion'
        if config['use_pseudo']: name += '_pseudo'
        if config['use_confidence_filter']: name += '_cf'
        key = f"{name}_{int(frac*100)}pct"
        
        if key in all_experiment_results:
            r = all_experiment_results[key]
            col_name = name.replace('ssl_', '+SSL').replace('motion', '+Motion').replace('pseudo', '+PL').replace('_cf', '+CF')
            row[col_name] = f"{r['mean']['Mean_Dice']:.4f}"
    
    table_b_rows.append(row)

table_b = pd.DataFrame(table_b_rows)
print("\n" + "="*80)
print("TABLE B: Label-Efficiency Results (Mean Dice)")
print("="*80)
print(table_b.to_string(index=False))
table_b.to_csv(os.path.join(TABLES_DIR, 'label_efficiency_all.csv'), index=False)

# Save all results
with open(os.path.join(RESULTS_DIR, 'finetune_results.json'), 'w') as f:
    serializable = {}
    for k, v in all_experiment_results.items():
        serializable[k] = {kk: vv for kk, vv in v.items() if kk != 'history'}
    json.dump(serializable, f, indent=2, default=str)

print("\n=== Step 7: Fine-Tuning COMPLETE ===")